# M7 L7: Retrieval and Ranking

**Build BM25, dense cosine, RRF, and MMR from scratch.  Hybrid wins.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

**Retrieval** means: given a question, find the most relevant documents to feed to the AI.

There are three main ways: **BM25** (count exact word matches — good for "SKU-12345"), **dense cosine** (compare the *meaning* — good for "kitten" vs "cat"), and **hybrid** (do both). Hybrid wins on most real-world collections.

When the top results are all saying the same thing, use **MMR** to pick a diverse set. When results are still not good enough, add a **reranker** — a slower, smarter model that reads the question and each result pair-by-pair.

## The 5 things to know

1. **BM25** catches exact matches; **dense** catches semantic similarity.
2. **Hybrid wins** — usually 10-20 points of recall better than either alone.
3. **RRF** is the standard merge — one line of code, no tuning.
4. **MMR** diversifies the result list; lambda controls the relevance/diversity trade-off.
5. **Cross-encoder reranker** is the next upgrade if recall@K is below target.

In [1]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm07-retrieval' / 'L7-retrieval-ranking.py'
_spec = importlib.util.spec_from_file_location('L7', _p)
_L7 = importlib.util.module_from_spec(_spec)
sys.modules['L7'] = _L7
_spec.loader.exec_module(_L7)

print(f'Setup OK. Lesson module: {_L7.LESSON_NUMBER}')

Setup OK. Lesson module: M7.L7


In [2]:
# Run the 10 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L7, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 20 tests...

  PASS  test_bm25_finds_exact_term
  PASS  test_bm25_idf_higher_for_rare_term
  PASS  test_bm25_returns_scores_in_range
  PASS  test_dense_finds_semantic_neighbor
  PASS  test_dense_score_is_bounded
  PASS  test_end_to_end_pipeline
  PASS  test_mmr_diversifies
  PASS  test_mmr_no_candidates_returns_empty
  PASS  test_mmr_respects_lambda
  PASS  test_retrieval_latency_ms_returns_stats
  PASS  test_retrieval_latency_ms_within_budget
  PASS  test_retrieval_math_invariants
  PASS  test_rrf_k_parameter_effect
  PASS  test_rrf_known_recipe
  PASS  test_rrf_merges_rankings
  PASS  test_throughput_decreases_with_latency
  PASS  test_throughput_per_second_inverse_of_latency
  PASS  test_tokenize_empty_string
  PASS  test_tokenize_lowercases
  PASS  test_tokenize_strips_punct

20/20 tests pass, 0 fail


In [3]:
# === Section 1: BM25 on the 4 sample queries ===
bm = _L7.BM25()
bm.fit(_L7.SAMPLE_DOCS)
print(f'{"query":<48}  top-3 (BM25)')
for q, _ in _L7.SAMPLE_QUERIES:
    hits = bm.score(q)[:3]
    ids = [i for i, _ in hits]
    print(f'  {q!r:<46}  {ids}')

query                                             top-3 (BM25)
  'how much vacation do I get'                    [1]
  '401k retirement match'                         [0]
  'what is the data retention policy'             [0, 6, 3]
  'how do I submit an expense'                    [3]


In [4]:
# === Section 2: dense cosine on the same queries ===
de = _L7.Dense()
de.fit(_L7.SAMPLE_DOCS)
print(f'{"query":<48}  top-3 (dense)')
for q, _ in _L7.SAMPLE_QUERIES:
    hits = de.score(q)[:3]
    ids = [i for i, _ in hits]
    print(f'  {q!r:<46}  {ids}')

query                                             top-3 (dense)
  'how much vacation do I get'                    [2, 4, 8]
  '401k retirement match'                         [8, 0, 4]
  'what is the data retention policy'             [1, 8, 3]
  'how do I submit an expense'                    [1, 8, 4]


In [5]:
# === Section 3: RRF (Reciprocal Rank Fusion) ===
print(f'{"query":<48}  top-5 (RRF)')
for q, _ in _L7.SAMPLE_QUERIES:
    bm_hits = bm.score(q)[:10]
    de_hits = de.score(q)[:10]
    merged = _L7.rrf([bm_hits, de_hits], k=_L7.RRF_K)[:5]
    ids = [i for i, _ in merged]
    print(f'  {q!r:<46}  {ids}')

query                                             top-5 (RRF)
  'how much vacation do I get'                    [1, 2, 4, 8, 3]
  '401k retirement match'                         [0, 8, 4, 7, 1]
  'what is the data retention policy'             [1, 3, 0, 6, 8]
  'how do I submit an expense'                    [3, 1, 8, 4, 2]


In [6]:
# === Section 4: RRF with weighted rankings (2x dense) ===
print(f'{"query":<48}  top-5 (RRF, 2x dense weight)')
for q, _ in _L7.SAMPLE_QUERIES:
    bm_hits = bm.score(q)[:10]
    de_hits = de.score(q)[:10]
    merged = _L7.rrf([bm_hits, de_hits, de_hits], k=_L7.RRF_K)[:5]
    ids = [i for i, _ in merged]
    print(f'  {q!r:<46}  {ids}')

query                                             top-5 (RRF, 2x dense weight)
  'how much vacation do I get'                    [1, 2, 4, 8, 3]
  '401k retirement match'                         [0, 8, 4, 7, 1]
  'what is the data retention policy'             [1, 3, 6, 0, 8]
  'how do I submit an expense'                    [3, 1, 8, 4, 2]


In [7]:
# === Section 5: MMR diversification (lambda=0.5) ===
import numpy as np
q, _ = _L7.SAMPLE_QUERIES[0]
qv = _L7.fake_embed(q)
cand_ids = [i for i, _ in de.score(q)[:20]]
selected = _L7.mmr(qv, de.vectors, cand_ids, lambda_=0.5, k=5)
print(f'query: {q!r}')
print(f'selected (in order): {selected}')
print(f'selected texts:')
for i in selected:
    print(f'  [{i}] {_L7.SAMPLE_DOCS[i][:60]!r}')

query: 'how much vacation do I get'
selected (in order): [2, 1, 9, 8, 4]
selected texts:
  [2] "Sick leave: 10 days per year, accrued monthly. Doctor's note"
  [1] 'Vacation policy: 20 days per year, accrued monthly at 1.67 d'
  [9] 'Hiring: phone screen, take-home, system design, behavioral; '
  [8] 'Vendor onboarding: security review, contract review, billing'
  [4] 'Code review: at least one engineer, PRs under 400 lines, com'


In [8]:
# === Section 6: MMR lambda sweep (diversity vs relevance) ===
print('Sweep lambda on MMR; report mean pairwise sim in top-5:\n')
for lam in (0.0, 0.3, 0.5, 0.7, 1.0):
    sel = _L7.mmr(qv, de.vectors, cand_ids, lambda_=lam, k=5)
    # mean pairwise sim in selected
    if len(sel) > 1:
        sims = []
        for a in sel:
            for b in sel:
                if a < b:
                    sims.append(float(np.dot(de.vectors[a], de.vectors[b])))
        mean_sim = sum(sims) / len(sims) if sims else 0
    else:
        mean_sim = 0
    print(f'  lambda={lam:.1f}  selected={sel}  mean_pair_sim={mean_sim:.3f}')

Sweep lambda on MMR; report mean pairwise sim in top-5:

  lambda=0.0  selected=[0, 1, 2, 5, 3]  mean_pair_sim=0.697
  lambda=0.3  selected=[2, 3, 6, 8, 5]  mean_pair_sim=0.683
  lambda=0.5  selected=[2, 1, 9, 8, 4]  mean_pair_sim=0.724
  lambda=0.7  selected=[2, 1, 8, 4, 9]  mean_pair_sim=0.724
  lambda=1.0  selected=[2, 4, 8, 1, 3]  mean_pair_sim=0.737


In [9]:
# === Section 7: recall@3 comparison ===
print('Mean recall@3 on the 4 sample queries:\n')
for label, fn in (
    ('BM25',  bm.score),
    ('Dense', de.score),
    ('RRF',   lambda q: _L7.rrf([bm.score(q), de.score(q)], k=_L7.RRF_K)),
):
    recall = 0
    for q, expected in _L7.SAMPLE_QUERIES:
        hits = fn(q)[:3]
        ids = [i for i, _ in hits]
        if int(expected) in ids:
            recall += 1
    print(f'  {label:<6}  recall@3 = {recall/len(_L7.SAMPLE_QUERIES):.2f}')

Mean recall@3 on the 4 sample queries:

  BM25    recall@3 = 1.00
  Dense   recall@3 = 0.25
  RRF     recall@3 = 0.75


## Architecture (Mermaid)

```mermaid
flowchart LR
    Q[query] --> BM[BM25<br/>sparse lexical]
    Q --> DE[Dense cosine<br/>semantic]
    BM --> T1[top-N BM25 hits]
    DE --> T2[top-N dense hits]
    T1 --> RRF[RRF merge<br/>1 / k+rank]
    T2 --> RRF
    RRF --> M[merged top-K]
    M --> MMR{MMR?<br/>diversify}
    MMR -->|no| OUT[final top-K]
    MMR -->|yes| DIV[diverse top-K]
    DIV --> OUT
```

**Read as:** Run both retrievers in parallel, merge with RRF, optionally diversify with MMR.

## Reflect (10 min)

1. Why does hybrid beat either BM25 or dense alone? Give a concrete example query for each.
2. Walk through RRF: if doc A is rank 1 in BM25 and rank 5 in dense, and doc B is rank 3 in BM25 and rank 1 in dense, which wins?
3. What does lambda=0.5 mean in MMR? What about lambda=0.0 and lambda=1.0?
4. When would you add a cross-encoder reranker? What's the cost?
5. (Stretch) Implement `weighted_rrf(rankings, weights, k)` that multiplies each ranking's contribution by a weight.

---

**Read next:**
- `lessons/m09-rag-e2e/L9-rag-end-to-end.ipynb` — the full RAG pipeline
- `lessons/m10-evaluation/L10-evaluation-ragas.ipynb` — RAGAS eval framework
- `SYLLABUS.md` — the 18 modules and 21-day plan